In [2]:
import pandas as pd

df = pd.read_csv("../data/transactions.csv", parse_dates=["timestamp"])
print(df.shape)
print(df["is_fraud"].value_counts())
print(df["is_fraud"].value_counts(normalize=True))

for col in ["category", "channel", "country", "new_device"]:
    print(df.groupby(col)["is_fraud"].mean().sort_values(ascending=False).round(4), "\n")

df["hour"] = df.timestamp.dt.hour
print(df.groupby("hour")["is_fraud"].mean().round(4))
print(df.groupby("is_fraud")["amount"].describe())

(200480, 9)
is_fraud
0    199400
1      1080
Name: count, dtype: int64
is_fraud
0    0.994613
1    0.005387
Name: proportion, dtype: float64
category
travel             0.0175
gift_cards         0.0173
electronics        0.0167
atm                0.0038
fuel               0.0031
restaurant         0.0031
grocery            0.0027
online_shopping    0.0027
Name: is_fraud, dtype: float64 

channel
online    0.0098
pos       0.0024
atm       0.0023
Name: is_fraud, dtype: float64 

country
SG    0.0097
AE    0.0074
UK    0.0062
US    0.0057
IN    0.0044
Name: is_fraud, dtype: float64 

new_device
1    0.0787
0    0.0031
Name: is_fraud, dtype: float64 

hour
0     0.0222
1     0.0133
2     0.0155
3     0.0173
4     0.0170
5     0.0018
6     0.0014
7     0.0032
8     0.0036
9     0.0034
10    0.0037
11    0.0024
12    0.0026
13    0.0015
14    0.0018
15    0.0021
16    0.0023
17    0.0016
18    0.0026
19    0.0015
20    0.0020
21    0.0012
22    0.0020
23    0.0035
Name: is_fraud, dtype: flo

In [2]:
from sklearn.metrics import average_precision_score, classification_report
from xgboost import XGBClassifier
import pandas as pd

df = pd.read_csv("../data/transactions.csv", parse_dates=["timestamp"])
df["hour"] = df.timestamp.dt.hour

cols = ["category", "channel", "country"]
X = pd.get_dummies(df[["amount", "hour", "new_device"] + cols], columns=cols)
y = df["is_fraud"]

# time-based split (random nahi, kyunki real life me past se future predict karte hain)
split = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

spw = (y_train == 0).sum() / (y_train == 1).sum()
model = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1,
                      scale_pos_weight=spw, eval_metric="aucpr", n_jobs=-1)
model.fit(X_train, y_train)

p = model.predict_proba(X_test)[:, 1]
print("PR-AUC:", round(average_precision_score(y_test, p), 4))
print("Random baseline:", round(y_test.mean(), 4))
print(classification_report(y_test, p > 0.5, digits=3))

PR-AUC: 0.1346
Random baseline: 0.0051
              precision    recall  f1-score   support

           0      0.998     0.922     0.958     39891
           1      0.041     0.659     0.078       205

    accuracy                          0.920     40096
   macro avg      0.520     0.790     0.518     40096
weighted avg      0.993     0.920     0.954     40096



In [3]:
import numpy as np
from sklearn.metrics import average_precision_score

df = df.sort_values(["user_id", "timestamp"]).reset_index(drop=True)
g = df.groupby("user_id")

# 1. user ke apne past average se amount ka ratio
past_avg = (g["amount"].cumsum() - df["amount"]) / g.cumcount()
df["amt_ratio"] = (df["amount"] / past_avg).fillna(1)

# 2. pichhle transaction se kitne seconds
df["secs_since_last"] = g["timestamp"].diff().dt.total_seconds().fillna(1e6)

# 3. velocity: pichhle 5 min / 1 hour me kitne transactions
for w in ["5min", "1h"]:
    c = df.groupby("user_id").rolling(w, on="timestamp")["amount"].count()
    df[f"tx_{w}"] = c.values - 1

# 4. home country sirf train period se (leakage nahi)
df = df.sort_values("timestamp").reset_index(drop=True)
split = int(len(df) * 0.8)
cutoff = df.timestamp.iloc[split]
home = df[df.timestamp < cutoff].groupby("user_id")["country"].agg(lambda s: s.mode()[0])
df["is_foreign"] = (df["country"] != df["user_id"].map(home).fillna(df["country"])).astype(int)

num = ["amount", "hour", "new_device", "amt_ratio",
       "secs_since_last", "tx_5min", "tx_1h", "is_foreign"]
X = pd.get_dummies(df[num + ["category", "channel"]], columns=["category", "channel"])
y = df["is_fraud"]

X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

spw = (y_train == 0).sum() / (y_train == 1).sum()
model = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1,
                      scale_pos_weight=spw, eval_metric="aucpr", n_jobs=-1)
model.fit(X_train, y_train)

p = model.predict_proba(X_test)[:, 1]
print("PR-AUC:", round(average_precision_score(y_test, p), 4), "(baseline 0.1346)")
print(classification_report(y_test, p > 0.5, digits=3))

PR-AUC: 0.4821 (baseline 0.1346)
              precision    recall  f1-score   support

           0      0.999     0.959     0.979     39891
           1      0.092     0.800     0.164       205

    accuracy                          0.958     40096
   macro avg      0.545     0.880     0.572     40096
weighted avg      0.994     0.958     0.975     40096



In [4]:
import numpy as np

FN_COST = 500   # missed fraud ka average loss (assumption)
FP_COST = 5     # false alarm ka cost: support call, customer friction (assumption)

yt = y_test.values
print("thr   alerts  caught  false_alarms  precision  recall  cost")
for t in [0.3, 0.5, 0.7, 0.8, 0.9, 0.95, 0.98]:
    pred = p > t
    tp = (pred & (yt == 1)).sum()
    fp = (pred & (yt == 0)).sum()
    fn = (~pred & (yt == 1)).sum()
    prec = tp / max(pred.sum(), 1)
    print(f"{t:<5} {pred.sum():<7} {tp:<7} {fp:<13} {prec:<10.3f} {tp/yt.sum():<7.3f} {fn*FN_COST + fp*FP_COST}")

# 3-tier decision policy
decision = np.select([p > 0.9, p > 0.5], ["BLOCK", "OTP_CHALLENGE"], "APPROVE")
print(pd.Series(decision).value_counts())
print(pd.crosstab(decision, yt))

thr   alerts  caught  false_alarms  precision  recall  cost
0.3   3157    178     2979          0.056      0.868   28395
0.5   1789    164     1625          0.092      0.800   28625
0.7   1131    148     983           0.131      0.722   33415
0.8   759     136     623           0.179      0.663   37615
0.9   444     117     327           0.264      0.571   45635
0.95  246     101     145           0.411      0.493   52725
0.98  124     81      43            0.653      0.395   62215
APPROVE          38307
OTP_CHALLENGE     1345
BLOCK              444
Name: count, dtype: int64
col_0              0    1
row_0                    
APPROVE        38266   41
BLOCK            327  117
OTP_CHALLENGE   1298   47


In [5]:
import joblib, json, os

BLOCK_THR, OTP_THR = 0.98, 0.5
decision = np.select([p > BLOCK_THR, p > OTP_THR],["BLOCK", "OTP_CHALLENGE"],"APPROVE")
print(pd.crosstab(decision, yt))

# save artifacts (will be loaded by the inference API and lambda)
os.makedirs("../models", exist_ok=True)
model.save_model("../models/fraud_xgb.json")
json.dump({"columns": list(X.columns), "block_thr": BLOCK_THR, "otp_thr": OTP_THR},
          open("../models/config.json", "w"))
home.to_json("../models/home_country.json")
print(os.listdir("../models"))

col_0              0   1
row_0                   
APPROVE        38266  41
BLOCK             43  81
OTP_CHALLENGE   1582  83
['fraud_xgb.json', 'config.json', 'home_country.json']
